In [ ]:
%%bash
# 1. Install system FFmpeg
apt-get update -y
apt-get install -y ffmpeg

# 2. Clone repository
cd /kaggle/working
if [ ! -d "SoulX-FlashHead" ]; then
    git clone https://github.com/Soul-AILab/SoulX-FlashHead.git
fi
cd SoulX-FlashHead

# 3. Install exact compatible versions for Kaggle Python 3.13 & CUDA
pip install "setuptools<82"
pip install "transformers==4.57.3" "diffusers==0.35.1" "huggingface-hub<1.0,>=0.34.0" "gradio==5.50.0"
pip install "xfuser>=0.4.3" "mediapipe<1.0.0"
pip install loguru pyloudnorm decord ftfy easydict imageio-ffmpeg yacs einops omegaconf librosa soundfile

In [ ]:
import os

# 1. Patch CPUFaceHandler with OpenCV fallback for Python 3.13
cpu_face_handler_path = "/kaggle/working/SoulX-FlashHead/flash_head/utils/cpu_face_handler.py"
cpu_face_handler_code = '''import cv2
import numpy as np
from typing import Tuple, List

try:
    import mediapipe as mp
    HAS_MP_SOLUTIONS = hasattr(mp, "solutions")
except ImportError:
    HAS_MP_SOLUTIONS = False


class CPUFaceHandler:
    """Handler for CPU-based face detection using MediaPipe with OpenCV fallback."""

    def __init__(self, model_selection: int = 1, min_detection_confidence: float = 0.0):
        self.use_mp = HAS_MP_SOLUTIONS
        if self.use_mp:
            self.detector = mp.solutions.face_detection.FaceDetection(
                model_selection=model_selection,
                min_detection_confidence=min_detection_confidence,
            )
        else:
            cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
            self.detector = cv2.CascadeClassifier(cascade_path)

    def detect(self, image: np.ndarray) -> Tuple[List[List[float]], List[float]]:
        bboxs, scores = [], []
        if self.use_mp:
            results = self.detector.process(image)
            detection_result = results.detections
            if detection_result is None:
                return bboxs, scores
            for detection in detection_result:
                bboxC = detection.location_data.relative_bounding_box
                x, y, w, h = bboxC.xmin, bboxC.ymin, bboxC.width, bboxC.height
                bboxs.append([x, y, x + w, y + h])
                scores.append(detection.score[0])
            return bboxs, scores
        else:
            img_h, img_w = image.shape[:2]
            gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
            faces = self.detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=4)
            if len(faces) == 0:
                bboxs.append([0.15, 0.15, 0.85, 0.85])
                scores.append(1.0)
                return bboxs, scores
            # Sort by largest face area
            faces = sorted(faces, key=lambda b: b[2] * b[3], reverse=True)
            for (x, y, w, h) in faces:
                bboxs.append([float(x / img_w), float(y / img_h), float((x + w) / img_w), float((y + h) / img_h)])
                scores.append(1.0)
            return bboxs, scores

    def __call__(self, image: np.ndarray) -> Tuple[List[List[float]], List[float]]:
        return self.detect(image)
'''

with open(cpu_face_handler_path, "w") as f:
    f.write(cpu_face_handler_code)

# 2. Patch flash_head_pipeline.py to ensure output_hidden_states=True
pipeline_path = "/kaggle/working/SoulX-FlashHead/flash_head/src/pipeline/flash_head_pipeline.py"
with open(pipeline_path, "r") as f:
    code = f.read()

if "use_return_dict=True" in code:
    code = code.replace("use_return_dict=True", "output_hidden_states=True, return_dict=True")
    with open(pipeline_path, "w") as f:
        f.write(code)

print("Patches applied successfully!")

In [ ]:
import os
from huggingface_hub import snapshot_download

base_dir = "/kaggle/working/SoulX-FlashHead"
models_dir = os.path.join(base_dir, "models")
os.makedirs(models_dir, exist_ok=True)

print("1/2: Downloading SoulX-FlashHead-1_3B...")
snapshot_download(
    repo_id="Soul-AILab/SoulX-FlashHead-1_3B",
    local_dir=os.path.join(models_dir, "SoulX-FlashHead-1_3B")
)

print("2/2: Downloading facebook/wav2vec2-base-960h (English Audio Encoder)...")
snapshot_download(
    repo_id="facebook/wav2vec2-base-960h",
    local_dir=os.path.join(models_dir, "wav2vec2-base-960h")
)

print("All model weights downloaded!")

In [ ]:
import os
import librosa
import soundfile as sf
from PIL import Image
from IPython.display import Audio, display

# ==============================================================================
# CHANGE THESE TWO PATHS TO YOUR UPLOADED KAGGLE DATASET FILES:
# Example: "/kaggle/input/my-flashhead-inputs/my_photo.jpg"
# ==============================================================================
RAW_IMAGE_PATH = "/kaggle/working/SoulX-FlashHead/examples/girl.png"
RAW_AUDIO_PATH = "/kaggle/working/SoulX-FlashHead/examples/podcast_sichuan_16k.wav"

# Optional: Set a max duration in seconds (e.g., 10.0 for a fast test, or None for full audio)
MAX_DURATION_SEC = None
# ==============================================================================

custom_dir = "/kaggle/working/custom_inputs"
os.makedirs(custom_dir, exist_ok=True)

PROCESSED_IMAGE_PATH = os.path.join(custom_dir, "input_portrait.png")
PROCESSED_AUDIO_PATH = os.path.join(custom_dir, "input_audio_16k.wav")

# 1. Process and save image as clean RGB PNG
img = Image.open(RAW_IMAGE_PATH).convert("RGB")
img.save(PROCESSED_IMAGE_PATH)
print(f"Saved portrait image: {PROCESSED_IMAGE_PATH} (Size: {img.size})")

# 2. Load audio, convert to mono 16kHz WAV (required by Wav2Vec2)
y, sr = librosa.load(RAW_AUDIO_PATH, sr=16000, mono=True, duration=MAX_DURATION_SEC)
sf.write(PROCESSED_AUDIO_PATH, y, 16000, subtype="PCM_16")

duration_sec = len(y) / 16000.0
estimated_chunks = int((duration_sec * 25) // 28) + 1
estimated_time_min = (estimated_chunks * 12.8) / 60.0

print(f"Saved 16kHz mono WAV: {PROCESSED_AUDIO_PATH}")
print(f"Audio Duration: {duration_sec:.2f} seconds")
print(f"Total Chunks to Generate: ~{estimated_chunks} chunks (~{estimated_time_min:.1f} minutes on T4)")

# Preview inputs
display(img.resize((256, int(256 * img.size[1] / img.size[0]))))
display(Audio(PROCESSED_AUDIO_PATH))

In [ ]:
%cd /kaggle/working/SoulX-FlashHead

!CUDA_VISIBLE_DEVICES=0 python generate_video.py \
    --ckpt_dir models/SoulX-FlashHead-1_3B \
    --wav2vec_dir models/wav2vec2-base-960h \
    --model_type lite \
    --cond_image /kaggle/working/custom_inputs/input_portrait.png \
    --audio_path /kaggle/working/custom_inputs/input_audio_16k.wav \
    --audio_encode_mode stream \
    --use_face_crop True \
    --save_file /kaggle/working/custom_output.mp4

In [ ]:
import os
import glob
from IPython.display import Video, FileLink, display

target_file = "/kaggle/working/custom_output.mp4"

if not os.path.exists(target_file):
    # Fallback search if generate_video.py saved to its default output folder
    candidates = [
        v for v in glob.glob("/kaggle/working/**/*.mp4", recursive=True)
        if "examples/" not in v and "models/" not in v and "assets/" not in v
    ]
    if candidates:
        target_file = max(candidates, key=os.path.getctime)

if os.path.exists(target_file):
    print(f"Video Ready: {target_file}")
    display(Video(target_file, embed=True, width=512))
    # Create relative path for Kaggle's FileLink download widget
    rel_path = os.path.relpath(target_file, "/kaggle/working")
    os.chdir("/kaggle/working")
    display(FileLink(rel_path))
else:
    print("Output video not found. Check Cell 5 logs for errors.")